# More advanced topics

## Fixing Unconstrained Parameters

Cosmologix uses a default set of cosmological parameters in its
computations: `{'Tcmb', 'Omega_bc', 'H0', 'Omega_b_h2', 'Omega_k', 'w',
'wa', 'm_nu', 'Neff'}`. However, certain combinations of cosmological
probes may be entirely insensitive to some of these parameters,
requiring their values to be fixed for the fitting process to
converge. For instance, the cosmic microwave background temperature
(`Tcmb`) is usually assumed constant in many analyses. Late-time
probes of the expansion history—like supernovae or uncalibrated baryon
acoustic oscillations (BAOs)—do not distinguish between baryon and
dark matter contributions (`Omega_b_h2`) or constrain the absolute
distance scale (`H0`), leaving these parameters effectively
unconstrained without additional data.

### Setting Fixed Parameters
In Cosmologix, you can fix parameters by passing the optional `fixed`
argument to the `fit` and `contours.frequentist_contour_2d_sparse`
functions. This mechanism also enables exploration of simplified
cosmological models, such as enforcing flatness (`Omega_k = 0`) or a
cosmological constant dark energy behavior (`w = -1`, `wa = 0`):

In [2]:
from cosmologix import distances, parameters, fitter, contours
import jax.numpy as jnp
from pprint import pprint

priors = parameters.get_priors(['planck2018', 'des5yr'])
fixed = {'Omega_k': 0.0, 'm_nu': 0.06, 'Neff': 3.046, 'Tcmb': 2.7255, 'wa': 0.0}
result = fitter.fit(priors, fixed=fixed)
grid = contours.frequentist_contour_2d_sparse(
    priors,
    grid={'Omega_bc': [0.18, 0.48, 30], 'w': [-0.6, -1.5, 30]},
    fixed=fixed
)

Using cached file: /home/marc/.cache/cosmologix/func_cache_DES5yr


Exploring contour ['Omega_bc', 'w'] (upper bound estimate):   7%|▏  | 27/382 [00:01<00:17, 20.26it/s]


### Degeneracy Checks
Recent versions of Cosmologix include a safeguard in the `fit`
function: it checks for perfect degeneracies among the provided priors
and fixed parameters before proceeding, raising an explicit error
message if any remain. The `contours.frequentist_contour_2d_sparse`
function, however, skips this check to allow exploration of partially
degenerate parameter combinations, offering flexibility for diagnostic
purposes.

### Command-Line Interface
From the command line, you can specify fixed parameters using the `-F`
or `--fixed` option, available for both `fit` and `explore`
commands. Additionally, the `-c` or `--cosmo` shortcut simplifies
restricting the model to predefined configurations (e.g., flat \( w
\)CDM):

In [3]:
! cosmologix fit -p DESIDR2 -F H0 70 -c FwCDM
! cosmologix explore Omega_bc w -p DESIDR2 -c FwCDM -F H0 70 -o desi_fwcdm.pkl

/usr/lib/python3.14/pty.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


Omega_bc = 0.2958 ± 0.0087
Omega_b_h2 = 0.0277 ± 0.0036
w = -0.915 ± 0.078
χ²=9.31 (d.o.f. = 10), χ²/d.o.f = 0.931
p-value: 50.28%
Exploring contour ['Omega_bc', 'w'] (upper bound estimate):  19%|▏| 100/515 [00:
Contour data saved to desi_fwcdm.pkl


### Automatic Parameter Fixing
For convenience, the `fit` command offers the `-A` or
`--auto-constrain` option, which automatically identifies and fixes
poorly constrained parameters. Use this with caution, as it may alter
the model by trimming parameters that lack sufficient constraints,
potentially affecting your results:

In [5]:
! cosmologix fit -p DES5yr -p DESIDR2 -A -c FwCDM

Using cached file: /home/marc/.cache/cosmologix/func_cache_DES5yr
Try again fixing H0
Omega_bc = 0.2959 ± 0.0089
Omega_b_h2 = 0.0257 ± 0.0018
w = -0.876 ± 0.039
M = -0.049 ± 0.011
χ²=1648.76 (d.o.f. = 1838), χ²/d.o.f = 0.897
p-value: 99.94%


## Cache Mechanism

Cosmologix includes a caching system to optimize performance by storing results from time-consuming operations. This mechanism applies to:
- Downloading external files, such as datasets.
- Expensive computations, like matrix inversions or factorizations used in \( \chi^2 \) calculations.
- Lengthy `jax.jit` compilations, which can have noticeable pre-run delays.

Caching helps reduce the initial overhead (sometimes called "preburn time") introduced by JAX’s just-in-time compilation and other resource-intensive tasks, making subsequent runs significantly faster.

### Accessing the Cache Directory
You can retrieve the location of the cache directory using the `tools` module:

In [6]:
from cosmologix import tools
print(tools.get_cache_dir())

/home/marc/.cache/cosmologix


This returns the path where cached files are stored, typically a platform-specific directory (e.g., `~/.cache/cosmologix` on Unix-like systems).

### Managing the Cache
If the cache grows too large or if you suspect outdated results are being loaded due to code changes, you can clear it entirely:

In [7]:
tools.clear_cache()

Cache directory /home/marc/.cache/cosmologix has been cleared.


This removes all cached files, forcing Cosmologix to recompute or redownload as needed on the next run. You can delete only the jit-compilation cache, avoiding the need to redownload all data with:

In [8]:
tools.clear_cache(jit=True)

# You can also perform the same operations from the command line:
! cosmologix clear-cache
! cosmologix clear-cache -j

Cache directory /home/marc/.cache/cosmologix/jit does not exist.
Cache directory /home/marc/.cache/cosmologix has been cleared.
Cache directory /home/marc/.cache/cosmologix/jit has been cleared.


### Notes

- The caching system is particularly useful for mitigating JAX’s compilation delays, but its effectiveness depends on consistent inputs and code stability.
- Use `clear_cache()` judiciously, as it deletes all cached data, including potentially large datasets, and will require internet connexion to download.
- Cache inflation is generally caused by the accumulation of compiled jit code for different combination of likelihoods and cosmologies. To avoid disk space issues the use of the persistent cache for jit code is deactivated when the cache size exceeds 1GB. 